# Sprint 4 Fine-Tuned Transformer Features

Thin Colab launcher for the `dl-final-project` Sprint 4 flow. Core logic lives in `src/` and `scripts/`.

This notebook is validation-only for Sprint 4. Do not compute test metrics here.

## Runtime

Use `Runtime > Change runtime type` and select a GPU before running full fine-tuning.

In [ ]:
!nvidia-smi

## Drive and Repository Setup

Expected Drive artifact root: `/content/drive/MyDrive/dl-final-artifact`.

The dataset split CSVs and HAM10000 images must resolve from the paths in the repo's dataset config or be copied/symlinked into the expected `data/` layout before running full jobs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/dl-final-artifact'
REPO_URL = 'https://github.com/KasimDeliaci/dl-final-project.git'
REPO_DIR = '/content/dl-final-project'

!mkdir -p "$DRIVE_ROOT"
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/dl-final-project
!git rev-parse --short HEAD

## Install Dependencies

In [ ]:
!pip -q install uv
!uv sync --dev

## Fast Verification

Run before the full GPU jobs.

In [ ]:
!PYTHONPATH=src uv run ruff check .
!PYTHONPATH=src uv run python -m pytest \
  tests/test_dataset_sprint1.py \
  tests/test_sprint2_features.py \
  tests/test_sprint3_fusion.py \
  tests/test_representation_complementarity.py \
  tests/test_sprint4_finetune.py

## Smoke Fine-Tuning Run

This checks the full code path without downloading pretrained weights or running the full dataset.

In [ ]:
!PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/finetune_backbones.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 \
  --epochs 1 \
  --batch-size 1 \
  --num-workers 0 \
  --limit-per-split 2 \
  --no-pretrained \
  --no-mixed-precision \
  --checkpoint-dir artifacts/checkpoints_smoke/ham10000/finetuned \
  --feature-root artifacts/features_smoke \
  --run-root artifacts/runs_smoke

## Full Sprint 4 Fine-Tuning

Run this only after dataset paths are valid in Colab. This produces train/validation fine-tuned feature caches and checkpoints. Test metrics are intentionally not computed.

In [ ]:
!PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/finetune_backbones.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 swin_tiny beit_base \
  --batch-size 16 \
  --num-workers 2

## Fine-Tuned Single-Backbone MLP

In [ ]:
!PYTHONPATH=src uv run python scripts/train_mlp.py \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned \
  --backbones vit_b16 swin_tiny beit_base \
  --batch-size 128

## Representative Fine-Tuned Fusion

In [ ]:
!PYTHONPATH=src uv run python scripts/run_fusion_matrix.py \
  --feature-source finetuned \
  --only-combination vit_b16 swin_tiny \
  --fusion-methods concat weighted_learned_512 weighted_pca_384 \
  --batch-size 128 \
  --run-tag s4_pair

!PYTHONPATH=src uv run python scripts/run_fusion_matrix.py \
  --feature-source finetuned \
  --only-combination vit_b16 swin_tiny beit_base \
  --fusion-methods concat weighted_learned_512 weighted_pca_384 \
  --batch-size 128 \
  --run-tag s4_triple

## Sync Generated Artifacts to Drive

Run after jobs complete. Generated checkpoints, features, and runs are intentionally not committed to Git.

In [ ]:
!mkdir -p "$DRIVE_ROOT/artifacts"
!rsync -av artifacts/ "$DRIVE_ROOT/artifacts/"